# TED API exploration

## Basic parameters and request function

Explore published procurement notices through TED's public Search API; no API key is required. Like the EUDAMED explorer, this notebook contains only setup, a request helper, and sample queries. TED searches use **POST with a JSON body**.

The URL and body parameters follow [request_body_parameters.md](request_body_parameters.md). [request_body_eample.txt](request_body_eample.txt) lists available fields; its `"string"` values are placeholders, not executable queries. [response_example.json](response_example.json) illustrates the response structure.

Official references: [Search API](https://docs.ted.europa.eu/api/latest/search.html) and [pagination](https://docs.ted.europa.eu/ODS/latest/reuse/search-api.html).


In [1]:
import pandas as pd
import requests

# Set the endpoint and a small default result projection.
SEARCH_URL = "https://api.ted.europa.eu/v3/notices/search"
FIELDS = [
    "publication-number",
    "publication-date",
    "notice-title",
    "buyer-name",
    "buyer-country",
    "main-classification-proc",
]
PAGE_SIZE = 5
TIMEOUT = 30


In [2]:
def ted_search(query, fields=None, page=1, limit=PAGE_SIZE, scope="ALL",
               check_query_syntax=False, only_latest_versions=False):
    """Search one page of published notices and return the full JSON response."""
    # Use page-number pagination; iterationNextToken is omitted in this mode.
    body = {
        "query": query,
        "fields": list(FIELDS if fields is None else fields),
        "page": page,
        "limit": limit,
        "scope": scope,
        "checkQuerySyntax": check_query_syntax,
        "paginationMode": "PAGE_NUMBER",
        "onlyLatestVersions": only_latest_versions,
    }
    response = requests.post(SEARCH_URL, json=body, timeout=TIMEOUT)
    response.raise_for_status()
    result = response.json()
    if result.get("timedOut"):
        raise RuntimeError("TED search timed out; narrow the query and retry.")
    return result


## Sample queries

Each call retrieves one small page. Edit the expert query, `fields`, `page`, `limit`, or `scope` as needed. Dates use `YYYYMMDD`; buyer country codes use three letters (Italy: `ITA`).

`limit` supports up to 250 notices, with at most 10,000 requested fields per page (`len(fields) * limit`). Page-number searches support up to 15,000 notices; use TED's iteration mode for larger extractions. `check_query_syntax=True` validates a query without retrieving notices.

The helper preserves `totalNoticeCount`, `notices`, and pagination metadata. DataFrames below keep nested multilingual values and lists intact. These examples only display results; they do not save files.


In [3]:
# Retrieve notices published on a fixed date for a reproducible first query.
notices = ted_search("publication-date = 20251001")
print("Total matching notices:", notices["totalNoticeCount"])
display(pd.DataFrame(notices["notices"]))


Total matching notices: 3589


,publication-number,publication-date,buyer-name,main-classification-proc,links,notice-title,buyer-country
0,639634-2025,2025-10-01+02:00,"{'lav': ['AS ""Sadales tīkls""']}",[45231400],{'xml': {'MUL': 'https://ted.europa.eu/en/noti...,{'hun': 'Lettország – Erősáramú vezeték építés...,[LVA]
1,639635-2025,2025-10-01+02:00,{'ell': ['ΔΗΜΟΣ ΚΑΙΣΑΡΙΑΝΗΣ']},[39293400],{'xml': {'MUL': 'https://ted.europa.eu/en/noti...,{'hun': 'Görögország – Műpázsit – ΠΡΟΜΗΘΕΙΑ: Α...,[GRC]
2,639636-2025,2025-10-01+02:00,{'eng': ['Sectoral Procurement Directorate']},[90731400],{'xml': {'MUL': 'https://ted.europa.eu/en/noti...,{'hun': 'Málta – A levegőszennyezés felügyelet...,[MLT]
3,639637-2025,2025-10-01+02:00,{'deu': ['DB InfraGO AG – Geschäftsbereich Fah...,[45234100],{'xml': {'MUL': 'https://ted.europa.eu/en/noti...,{'hun': 'Németország – Vasútépítés – Kabeltief...,[DEU]
4,639638-2025,2025-10-01+02:00,{'spa': ['Ajuntament de Santa Eulàlia de Ronça...,[55523100],{'xml': {'MUL': 'https://ted.europa.eu/en/noti...,{'hun': 'Spanyolország – Iskolai ebédszolgálta...,[ESP]


In [4]:
# Filter the same publication date to Italian buyers.
italian_notices = ted_search(
    "publication-date = 20251001 AND buyer-country = ITA"
)
print("Total matching notices:", italian_notices["totalNoticeCount"])
display(pd.DataFrame(italian_notices["notices"]))


Total matching notices: 88


,publication-number,publication-date,buyer-name,main-classification-proc,links,notice-title,buyer-country
0,639677-2025,2025-10-01+02:00,{'ita': ['DIPARTIMENTO TUTELA AMBIENTALE']},[34130000],{'xml': {'MUL': 'https://ted.europa.eu/en/noti...,{'hun': 'Olaszország – Áruszállító gépjárművek...,[ITA]
1,639707-2025,2025-10-01+02:00,{'ita': ['Consorzio per le Autostrade Sicilian...,[90900000],{'xml': {'MUL': 'https://ted.europa.eu/en/noti...,{'hun': 'Olaszország – Takarítási és higiéniai...,[ITA]
2,639762-2025,2025-10-01+02:00,{'ita': ['Trenord S.r.l.']},[50000000],{'xml': {'MUL': 'https://ted.europa.eu/en/noti...,{'hun': 'Olaszország – Javítási és karbantartá...,[ITA]
3,639973-2025,2025-10-01+02:00,{'ita': ['ASST SPEDALI CIVILI DI BRESCIA']},[33194120],{'xml': {'MUL': 'https://ted.europa.eu/en/noti...,{'hun': 'Olaszország – Infúziós kellékek – PRO...,[ITA]
4,639988-2025,2025-10-01+02:00,"{'deu': ['Brenner Basistunnel BBT SE'], 'ita':...",[72263000],{'xml': {'MUL': 'https://ted.europa.eu/en/noti...,{'hun': 'Olaszország – Szoftvermegvalósítási s...,[ITA]


In [5]:
# Search 2025 notices for ophthalmology equipment (CPV 33122000).
ophthalmology_notices = ted_search(
    "publication-date >= 20250101 AND publication-date <= 20251231 "
    "AND classification-cpv = 33122000"
)
print("Total matching notices:", ophthalmology_notices["totalNoticeCount"])
display(pd.DataFrame(ophthalmology_notices["notices"]))


Total matching notices: 528


,publication-number,publication-date,buyer-name,main-classification-proc,links,notice-title,buyer-country
0,881-2025,2025-01-02+01:00,{'slv': ['ZDRAVSTVENI DOM DR. ADOLFA DROLCA MA...,[33122000],{'xml': {'MUL': 'https://ted.europa.eu/en/noti...,{'hun': 'Szlovénia – Szemészeti berendezések –...,[SVN]
1,13398-2025,2025-01-09+01:00,{'ron': ['JUDEȚUL SUCEAVA']},[33100000],{'xml': {'MUL': 'https://ted.europa.eu/en/noti...,{'hun': 'Románia – Orvosi felszerelések – Furn...,[ROU]
2,15574-2025,2025-01-10+01:00,"{'ron': ['INSTITUTUL DE PSIHIATRIE ""SOCOLA"" IA...",[33158210],{'xml': {'MUL': 'https://ted.europa.eu/en/noti...,{'hun': 'Románia – Stimulátorok – ACHIZITIA DE...,[ROU]
3,20707-2025,2025-01-13+01:00,{'ron': ['SPITALUL ORASENESC MOLDOVA NOUA']},[33100000],{'xml': {'MUL': 'https://ted.europa.eu/en/noti...,{'hun': 'Románia – Orvosi felszerelések – Echi...,[ROU]
4,22318-2025,2025-01-14+01:00,{'deu': ['Südtiroler Sanitätsbetrieb - Abteilu...,[33122000],{'xml': {'MUL': 'https://ted.europa.eu/en/noti...,{'hun': 'Olaszország – Szemészeti berendezések...,[ITA]
